<a href="https://www.kaggle.com/code/tarzon/forest-fire-prediction?scriptVersionId=185959697" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Forest Fire Prediction

**Introduction**


Welcome to this notebook on forest fire prediction! In this notebook, we will explore a dataset containing information about forest fires and predict the burned area using various features. Let's dive in!

I was shared detailed of the code in article, Please visit the below link
https://medium.com/@parulrajput27/forest-fire-prediction-a-comprehensive-guide-for-beginners-b96fb7060c7f

**importing libraries**

First, we need to import the necessary libraries for our analysis.

In [ ]:
    #importing library
    import pandas as pd
    import numpy as np
    import seaborn as sns
    import matplotlib.pyplot as plt
    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, confusion_matrix

**Reading dataset**

Let's load the dataset and take a quick look at the first few rows.

In [ ]:
# Load the data
data = pd.read_csv('/kaggle/input/forest-fires-data-set/forestfires.csv')

# Display the first few rows
data.head()

In [ ]:
#dataset  shape
data.shape

In [ ]:
# Basic info
data.info()

In [ ]:
# Statistical summary
data.describe()

In [ ]:
#Handle missing values
data.isnull().sum()

The dataset does not have missing values. Now let's see if the values seem to be reasonable.

In [ ]:
# Check for duplicate values
data.duplicated().sum()

In [ ]:
# Drop duplicate values
data.drop_duplicates()

**Exploring the Dataset**

Now, let's do some exploratory data analysis (EDA) to understand the dataset better.


**Univariate Analysis**

We'll start with some univariate analysis to look at the distribution of individual features.

In [ ]:
# Plotting the distribution of 'area' (our target variable)
plt.figure(figsize=(10, 6))
sns.histplot(data['area'], bins=30, kde=True)
plt.title('Distribution of Burned Area')
plt.xlabel('Burned Area')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Plotting the distribution of temperature
plt.figure(figsize=(10, 6))
sns.histplot(data['temp'], bins=30, kde=True)
plt.title('Distribution of Temperature')
plt.xlabel('Temperature')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Plotting the Count of Forest Fires by month
plt.figure(figsize=(12, 6))
sns.countplot(x='month', data=data)
plt.title('Count of Forest Fires by Month')
plt.xlabel('Month')
plt.ylabel('Count')
plt.show()

**Bivariate Analysis**


Next, we'll explore the relationship between different pairs of features.

In [ ]:
# Box plot of Burned Area by Month

plt.figure(figsize=(12, 6))
sns.boxplot(x='month', y='area', data=data)
plt.title('Burned Area by Month')
plt.xlabel('Month')
plt.ylabel('Burned Area (ha)')
plt.show()

In [ ]:
# Scatter plot of temperature vs. area
plt.figure(figsize=(10, 6))
sns.scatterplot(x='temp', y='area', data=data)
plt.title('Temperature vs. Burned Area')
plt.xlabel('Temperature')
plt.ylabel('Burned Area')
plt.show()

In [ ]:
# Scatter plot of wind vs. area
plt.figure(figsize=(10, 6))
sns.scatterplot(x='wind', y='area', data=data)
plt.title('Wind vs. Burned Area')
plt.xlabel('Wind')
plt.ylabel('Burned Area')
plt.show()


**Multivariate Analysis**

In [ ]:
# Define the damage categories
def categorize_damage(area):
    if area == 0:
        return 'No Damage'
    elif area <= 1:
        return 'Low Damage'
    elif area <= 10:
        return 'Moderate Damage'
    else:
        return 'High Damage'

# Apply the function to create a new column 'damage_category'
data['damage_category'] = data['area'].apply(categorize_damage)

# List of features to analyze
features = ['temp', 'RH', 'wind']

# Plotting the distribution of features across damage categories
plt.figure(figsize=(18, 12))

for i, feature in enumerate(features, 1):
    plt.subplot(3, 1, i)
    sns.boxplot(x='damage_category', y=feature, data=data, palette='viridis')
    plt.title(f'{feature.capitalize()} by Damage Category')
    plt.xlabel('Damage Category')
    plt.ylabel(feature.capitalize())

plt.tight_layout()
plt.show()


In [ ]:
# Pair Plot of Fire Weather Indicators
sns.pairplot(data[['FFMC', 'DMC', 'DC', 'ISI', 'temp', 'RH', 'wind', 'rain', 'area']], diag_kind='kde')
plt.suptitle('Pairplot of Fire Weather Indicators', y=1.02)
plt.show()

**Outlier Detection**


Let's check for outliers in the dataset.

In [ ]:
# Box plot to identify outliers in the 'area' feature
plt.figure(figsize=(10, 6))
sns.boxplot(x='area', data=data)
plt.title('Box Plot of Burned Area')
plt.xlabel('Burned Area')
plt.show()

In [ ]:
# Box plot to identify outliers in the 'temp' feature
plt.figure(figsize=(10, 6))
sns.boxplot(x='temp', data=data)
plt.title('Box Plot of Temperature')
plt.xlabel('Temperature')
plt.show()

**Feature Engineering**


Now, we'll create some new features from the existing ones to improve our model's performance.

In [ ]:
# Convert 'month' and 'day' to numerical values
data['month'] = data['month'].map({'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6, 'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12})
data['day'] = data['day'].map({'mon': 1, 'tue': 2, 'wed': 3, 'thu': 4, 'fri': 5, 'sat': 6, 'sun': 7})

# Create a new feature 'is_weekend'
data['is_weekend'] = data['day'].apply(lambda x: 1 if x in [6, 7] else 0)


**Model building**

In [ ]:
from sklearn.model_selection import train_test_split
X = data.drop(['area','damage_category'],axis=1)
y = data['area']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.35,random_state=200)

In [ ]:
# Initialize the model
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, random_state=42)

# Train the model
model.fit(X_train, y_train)


In [ ]:
# Make predictions
y_pred = model.predict(X_test)

In [ ]:
# Evaluate the model
from sklearn.metrics import mean_squared_error, r2_score

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'Mean Squared Error: {mse}')
print(f'R2 Score: {r2}')

In [ ]:
# Plotting actual vs predicted values
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.xlabel('Actual Burned Area')
plt.ylabel('Predicted Burned Area')
plt.title('Actual vs Predicted Burned Area')
plt.plot([0, max(y_test)], [0, max(y_pred)], color='red', linestyle='--')
plt.show()

We hope you found this notebook informative, and helpful in understanding the basics of using machine learning for predicting forest fires.

Your feedback is invaluable! Please feel free to share your thoughts, comments, and suggestions on how we can improve this notebook. Whether it's about the code, the explanations, or any additional topics you'd like to see covered, we're eager to hear from you.

# Happy Coding